In [1]:
import pandas as pd
import json

In [4]:
import glob
files = glob.glob("../data/external/*V2*instruct*.json")
files

['../data/external/first_100_failing_examples_without_docstrings_base_model_og_prompt_V2_with_instruct_response.json',
 '../data/external/first_100_passing_examples_without_docstrings_base_model_og_prompt_V2_with_instruct_response.json']

In [5]:
failing_file = files[0]
passing_file = files[1]

### Mapping criteria.
- Filter cases where the base model output contains docstrings OR base model has 'return 0' or empty response with any return in output.
- Check whether the output differs after the function signature only, mark the output position where the outputs start to diffe in exact.
- Check which token position does that string position refer to in both base and instruct outputs.

In [7]:
def create_input_prompt_prefix(task_prompt, test_list, mode = "base"):
    prompt = (
            "You are an expert Python programmer, and here is your task: "
            f"{task_prompt} Your code should pass these tests:\n\n"
            + "\n".join(test_list) + "\nWrite your code below starting with \"```python\" and ending with \"```\".\n```python\n"
        )
    if mode == "instruct":
        prompt = (
            "You are an expert Python programmer, and here is your task: "
            f"{task_prompt} Your code should pass these tests:\n\n"
            + "\n".join(test_list) + "\nWrite your code, without docstrings, below starting with \"```python\" and ending with \"```\".\n```python\n"
        )
    
    return prompt

In [8]:
def filter_base_output(base_output):
    if "return 0" in base_output:
        return False
    if base_output.strip() == "":
        return False
    if '"""' in base_output or "'''" in base_output:
        return False
    return True

In [16]:
import re

def check_if_different_after_signature(base_output, instruct_output):
    """
    Returns True if the bodies (after function signature) differ.
    Returns False if they are identical.
    """

    def strip_signature(code):
        # Match: def <name>(params):
        pattern = r"def\s+\w+\s*\([^)]*\)\s*:"
        match = re.search(pattern, code)
        if not match:
            return code  # No signature found — treat full text as body
        return code[match.end():].strip()

    base_body = strip_signature(base_output)
    instruct_body = strip_signature(instruct_output)

    # if base_body == instruct_body:
    #     print("Bodies are identical:")
    #     print(base_body)

    return base_body != instruct_body

In [9]:
all_task_list = []
for file in [failing_file, passing_file]:
    with open(file, "r") as f:
        data = json.load(f)
    all_task_list.extend(data)

len(all_task_list), all_task_list[0].keys()

(100,
 dict_keys(['source_file', 'task_id', 'prompt', 'code', 'test_imports', 'test_list', 'model_output', 'instruct_code']))

In [10]:
filtered_task_list = [task for task in all_task_list if filter_base_output(task['model_output'])]

len(filtered_task_list), filtered_task_list[0].keys()

(89,
 dict_keys(['source_file', 'task_id', 'prompt', 'code', 'test_imports', 'test_list', 'model_output', 'instruct_code']))

In [17]:
different_answer_list = [task for task in filtered_task_list if check_if_different_after_signature(task['model_output'], task['instruct_code'])]
same_answer_list = [task for task in filtered_task_list if not check_if_different_after_signature(task['model_output'], task['instruct_code'])]

len(different_answer_list), different_answer_list[0].keys()

(81,
 dict_keys(['source_file', 'task_id', 'prompt', 'code', 'test_imports', 'test_list', 'model_output', 'instruct_code']))

In [18]:
len(same_answer_list), same_answer_list[0].keys()

(8,
 dict_keys(['source_file', 'task_id', 'prompt', 'code', 'test_imports', 'test_list', 'model_output', 'instruct_code']))

In [22]:
import re

def find_first_position_of_difference(base_output, instruct_output):
    def strip_signature(code):
        pattern = r"def\s+\w+\s*\([^)]*\)\s*:"
        match = re.search(pattern, code)
        if not match:
            return code, 0  # no signature, full code is body
        return code[match.end():], match.end()

    # Strip signatures but also track offset into original strings
    base_body, base_offset = strip_signature(base_output)
    instruct_body, instruct_offset = strip_signature(instruct_output)

    # Compare bodies
    min_len = min(len(base_body), len(instruct_body))
    for i in range(min_len):
        if base_body[i] != instruct_body[i]:
            # Found difference: return position in original strings
            return base_offset + i, instruct_offset + i

    # If one is longer than the other, difference starts at min_len
    if len(base_body) != len(instruct_body):
        return base_offset + min_len, instruct_offset + min_len

    # No difference
    return None

def find_first_position_of_difference_in_full_output(base_output, instruct_output, test_list, task_prompt):
    base_input_prefix = create_input_prompt_prefix(task_prompt, test_list, mode="base")
    instruct_input_prefix = create_input_prompt_prefix(task_prompt, test_list, mode="instruct")
    ans = find_first_position_of_difference(base_output, instruct_output)
    if ans is not None:
        base_pos, instruct_pos = ans
        # Adjust positions to be relative to full output including input prompt
        return base_pos + len(base_input_prefix), instruct_pos + len(instruct_input_prefix)
    return None

In [23]:
different_answer_list[0].keys()

dict_keys(['source_file', 'task_id', 'prompt', 'code', 'test_imports', 'test_list', 'model_output', 'instruct_code'])

In [25]:
find_first_position_of_difference_in_full_output(different_answer_list[0]['model_output'], different_answer_list[0]['instruct_code'], different_answer_list[0]['test_list'], different_answer_list[0]['prompt'])

(395, 411)

In [26]:
from transformers import AutoTokenizer

gemma2_2b_tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b")
gemma2_2b_it_tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b-it")

def char_pos_to_token_pos(text, tokenizer, char_pos):
    """
    Convert a character offset (char_pos) in a string to the token index.
    """
    encoded = tokenizer(
        text,
        return_offsets_mapping=True,
        add_special_tokens=False
    )
    offsets = encoded["offset_mapping"]

    for tok_idx, (start, end) in enumerate(offsets):
        if start <= char_pos < end:
            return tok_idx

    # If the differing character is beyond all tokens (rare but possible)
    return len(offsets) - 1


def find_first_token_position_difference(entry):
    base_output = entry['model_output']
    instruct_output = entry['instruct_code']

    # Build full rendered prompt+output strings
    full_base_output = create_input_prompt_prefix(
        entry['prompt'], entry['test_list'], mode="base"
    ) + base_output

    full_instruct_output = create_input_prompt_prefix(
        entry['prompt'], entry['test_list'], mode="instruct"
    ) + instruct_output

    # Step 1: find char positions
    pos = find_first_position_of_difference_in_full_output(
        base_output,
        instruct_output,
        entry['test_list'],
        entry['prompt']
    )

    if pos is None:
        return None

    base_char_pos, instruct_char_pos = pos

    # Step 2: map char pos → token pos
    base_tok_ids = gemma2_2b_tokenizer(full_base_output).input_ids
    instruct_tok_ids = gemma2_2b_it_tokenizer(full_instruct_output).input_ids

    base_token_pos = char_pos_to_token_pos(
        full_base_output, gemma2_2b_tokenizer, base_char_pos
    )
    instruct_token_pos = char_pos_to_token_pos(
        full_instruct_output, gemma2_2b_it_tokenizer, instruct_char_pos
    )

    return {
        "base_char_pos": base_char_pos,
        "instruct_char_pos": instruct_char_pos,
        "base_token_pos": base_token_pos,
        "instruct_token_pos": instruct_token_pos,
        "base_token_id": base_tok_ids[base_token_pos],
        "instruct_token_id": instruct_tok_ids[instruct_token_pos]
    }

In [27]:
find_first_token_position_difference(different_answer_list[0])

{'base_char_pos': 395,
 'instruct_char_pos': 411,
 'base_token_pos': 115,
 'instruct_token_pos': 121,
 'base_token_id': 648,
 'instruct_token_id': 552}

In [28]:
task_with_pos_entry = []
for task in different_answer_list:
    pos_info = find_first_token_position_difference(task)
    if pos_info is not None:
        task_with_pos_entry.append({
            **task,
            "position_info": pos_info
        })
        
len(task_with_pos_entry), task_with_pos_entry[0].keys()

(81,
 dict_keys(['source_file', 'task_id', 'prompt', 'code', 'test_imports', 'test_list', 'model_output', 'instruct_code', 'position_info']))

In [29]:
task_with_pos_entry[0]['position_info']

{'base_char_pos': 395,
 'instruct_char_pos': 411,
 'base_token_pos': 115,
 'instruct_token_pos': 121,
 'base_token_id': 648,
 'instruct_token_id': 552}

In [30]:
task_with_pos_entry[1]['position_info']

{'base_char_pos': 511,
 'instruct_char_pos': 526,
 'base_token_pos': 133,
 'instruct_token_pos': 138,
 'base_token_id': 141,
 'instruct_token_id': 141}

In [32]:
with open("../data/external/first_100_selected_examples_without_docstrings_base_model_og_prompt_V2_instruct_comparison_with_position_info.json", "w") as f:
    json.dump(task_with_pos_entry, f, indent=4)

In [33]:
## inspect a sample from the saved file.

sample = task_with_pos_entry[15]

In [40]:
sample.keys()

dict_keys(['source_file', 'task_id', 'prompt', 'code', 'test_imports', 'test_list', 'model_output', 'instruct_code', 'position_info'])

In [34]:
full_instruct_response = create_input_prompt_prefix(
    sample['prompt'], sample['test_list'], mode="instruct"
) + sample['instruct_code']
full_base_response = create_input_prompt_prefix(
    sample['prompt'], sample['test_list'], mode="base"
) + sample['model_output']

In [38]:
print(full_instruct_response)

You are an expert Python programmer, and here is your task: Write a function that counts the number of pairs of integers in a list that xor to an even number. Your code should pass these tests:

assert find_even_pair([5, 4, 7, 2, 1]) == 4
assert find_even_pair([7, 2, 8, 1, 0, 5, 11]) == 9
assert find_even_pair([1, 2, 3]) == 1
Write your code, without docstrings, below starting with "```python" and ending with "```".
```python
def find_even_pair(nums):
  count = 0
  for i in range(len(nums)):
    for j in range(i + 1, len(nums)):
      if (nums[i] ^ nums[j]) % 2 == 0:
        count += 1
  return count



In [39]:
print(full_base_response)

You are an expert Python programmer, and here is your task: Write a function that counts the number of pairs of integers in a list that xor to an even number. Your code should pass these tests:

assert find_even_pair([5, 4, 7, 2, 1]) == 4
assert find_even_pair([7, 2, 8, 1, 0, 5, 11]) == 9
assert find_even_pair([1, 2, 3]) == 1
Write your code below starting with "```python" and ending with "```".
```python
def find_even_pair(list):
    even_pairs = 0
    for i in range(len(list)):
        for j in range(i+1, len(list)):
            if list[i] ^ list[j] == 0:
                even_pairs += 1
    return even_pairs


In [42]:
def detokenize(tokenizer, token_ids):
    """
    Convert a list of token IDs back into a string.
    """
    return tokenizer.decode(token_ids, skip_special_tokens=False)

In [43]:
def tokenize(tokenizer, text):
    """
    Convert a string into a list of token IDs.
    """
    return tokenizer.encode(text, add_special_tokens=False)

In [46]:
base_ids = tokenize(gemma2_2b_tokenizer, full_base_response)
instruct_ids = tokenize(gemma2_2b_it_tokenizer, full_instruct_response)

In [47]:
base_ids[sample["position_info"]["base_token_pos"]]

141

In [54]:
detokenize(gemma2_2b_tokenizer, [base_ids[sample["position_info"]["base_token_pos"] + 1]])

'even'

In [48]:
instruct_ids[sample["position_info"]["instruct_token_pos"]]

1656

In [53]:
detokenize(gemma2_2b_it_tokenizer, [instruct_ids[sample["position_info"]["instruct_token_pos"]]])

'count'

In [56]:
sample = task_with_pos_entry[10]
full_base_response = create_input_prompt_prefix(
    sample['prompt'], sample['test_list'], mode="base"
) + sample['model_output']
full_instruct_response = create_input_prompt_prefix(
    sample['prompt'], sample['test_list'], mode="instruct"
) + sample['instruct_code']

base_ids = tokenize(gemma2_2b_tokenizer, full_base_response)
instruct_ids = tokenize(gemma2_2b_it_tokenizer, full_instruct_response)

diff_base_id = base_ids[sample["position_info"]["base_token_pos"]]
diff_instruct_id = instruct_ids[sample["position_info"]["instruct_token_pos"]]
detokenized_base_token = detokenize(gemma2_2b_tokenizer, [diff_base_id])
detokenized_instruct_token = detokenize(gemma2_2b_it_tokenizer, [diff_instruct_id])

print(full_base_response)
print(full_instruct_response)
print(f"Base token ID: {diff_base_id}, Detokenized: {detokenized_base_token}")
print(f"Instruct token ID: {diff_instruct_id}, Detokenized: {detokenized_instruct_token}")

You are an expert Python programmer, and here is your task: Write a function to calculate whether the matrix is a magic square. Your code should pass these tests:

assert magic_square_test([[7, 12, 1, 14], [2, 13, 8, 11], [16, 3, 10, 5], [9, 6, 15, 4]])==True
assert magic_square_test([[2, 7, 6], [9, 5, 1], [4, 3, 8]])==True
assert magic_square_test([[2, 7, 6], [9, 5, 1], [4, 3, 7]])==False
Write your code below starting with "```python" and ending with "```".
```python
def magic_square(matrix):
    # Your code here
    return True
You are an expert Python programmer, and here is your task: Write a function to calculate whether the matrix is a magic square. Your code should pass these tests:

assert magic_square_test([[7, 12, 1, 14], [2, 13, 8, 11], [16, 3, 10, 5], [9, 6, 15, 4]])==True
assert magic_square_test([[2, 7, 6], [9, 5, 1], [4, 3, 8]])==True
assert magic_square_test([[2, 7, 6], [9, 5, 1], [4, 3, 7]])==False
Write your code, without docstrings, below starting with "```python" a